# Portfolio Optimization using QAOA from Scratch

This notebook is a guided, beginner-friendly walkthrough of how portfolio optimization can be written as a small quantum optimization problem and solved with the Quantum Approximate Optimization Algorithm (QAOA).

**Learning path:**
1. Describe a portfolio as a binary string, where `1` means we select an asset and `0` means we leave it out.
2. Turn the finance goal into a QUBO objective that balances return against risk.
3. Convert the QUBO into an Ising Hamiltonian that a quantum circuit can measure.
4. Build a QAOA circuit, tune its parameters with a classical optimizer, and read the most likely bitstrings as candidate portfolios.

The goal is not to build a production trading system. The goal is to understand the complete logic chain: finance objective -> binary model -> quantum Hamiltonian -> QAOA circuit -> interpretable portfolio result.

**How to use this notebook:** run the cells from top to bottom first, then return to earlier cells and change one variable at a time. This makes it easier to see which change affected the final portfolio probabilities.


## 1. Mathematical Formulation

Portfolio optimization asks a simple question: **which assets should we select so that the portfolio has good expected return without taking too much risk?**

We represent the decision with a binary vector $x$:
- $x_i = 1$ means asset $i$ is selected.
- $x_i = 0$ means asset $i$ is not selected.

The notebook uses the Markowitz mean-variance idea. In this notation, $\mu$ stores the expected return of each asset, $\Sigma$ stores how risky the assets are together, and $q$ controls how strongly we punish risk.

The cost function is:
$$ \min_{x \in \{0,1\}^n} \frac{q}{2} x^T \Sigma x - \mu^T x $$

Read this as: **minimize risk minus reward**. A lower cost is better because the optimizer is encouraged to reduce risk while still choosing assets with high expected return.

A useful mental model: `q` is the caution dial. Larger `q` means the model worries more about risk; smaller `q` means it pays more attention to expected return.


## 2. QUBO and Ising Mapping

The formula above is a QUBO: a **Quadratic Unconstrained Binary Optimization** problem. "Binary" means every decision variable is either 0 or 1, and "quadratic" means the objective can include pairs of variables such as $x_i x_j$.

Quantum circuits naturally work with qubits. To connect the binary variables to qubits, we rewrite each binary variable using a Pauli-Z value:
$$ x_i = \frac{1 - Z_i}{2} $$

Here, $Z_i$ has possible measured values $+1$ and $-1$. This substitution converts the portfolio cost into an **Ising Hamiltonian**, also called the cost Hamiltonian $H_C$.

That conversion matters because QAOA does not optimize the original formula directly. It optimizes a quantum circuit whose measured energy matches the cost we care about.


## 3. Libraries and Packages

This section imports the tools used in the rest of the notebook.

- `numpy` handles vectors and matrices such as returns and covariance.
- `scipy.optimize` runs the classical optimizer that adjusts QAOA parameters.
- `matplotlib` plots the final portfolio probabilities.
- `qiskit` builds the quantum circuit and simulates the final quantum state.

QAOA is a hybrid algorithm, so it needs both classical numerical tools and quantum circuit tools.


In [ ]:
import numpy as np
import scipy.optimize as opt
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit.circuit import Parameter


## 4. Generate Mock Financial Data

To keep the first version easy to follow, the notebook uses only three mock assets.

For each asset we define:
- an expected return, stored in `mu`;
- risk and co-movement with other assets, stored in the covariance matrix `Sigma`.

The value `q = 0.5` is the risk-aversion factor. If `q` is increased, the optimizer becomes more cautious about risky combinations. If `q` is decreased, expected return has more influence.


In [ ]:
# Number of assets
n_assets = 3

# Expected returns
mu = np.array([0.1, 0.2, 0.15])

# Covariance matrix (must be symmetric positive semi-definite)
Sigma = np.array([
    [0.05, 0.01, 0.02],
    [0.01, 0.06, 0.03],
    [0.02, 0.03, 0.04]
])

# Risk factor
q = 0.5


## 5. Build Ising Hamiltonian

Now we translate the portfolio objective into quantum language.

Starting from the substitution $x_i = \frac{1 - Z_i}{2}$, the cost becomes:
$$ \frac{q}{2} \left(\frac{1-Z}{2}\right)^T \Sigma \left(\frac{1-Z}{2}\right) - \mu^T \left(\frac{1-Z}{2}\right) $$

After expanding the expression, we collect two types of terms:
- single-qubit terms such as $Z_i$, which describe the effect of selecting one asset;
- two-qubit terms such as $Z_i Z_j$, which describe how pairs of assets interact through covariance.

Constant terms are ignored because they shift every portfolio score by the same amount. They do not change which portfolio has the lowest cost.


In [ ]:
# Build the Pauli operators for the cost Hamiltonian
linear_z = np.zeros(n_assets)
quadratic_zz = np.zeros((n_assets, n_assets))

for i in range(n_assets):
    linear_z[i] += mu[i] * 0.5
    for j in range(n_assets):
        Q_ij = q * Sigma[i, j] / 2
        linear_z[i] -= Q_ij * 0.5 * 2
        quadratic_zz[i, j] += Q_ij * 0.25

obs_terms = []
for i in range(n_assets):
    if abs(linear_z[i]) > 1e-8:
        pauli_str = ['I'] * n_assets
        pauli_str[n_assets - 1 - i] = 'Z'
        obs_terms.append(("".join(pauli_str), linear_z[i]))

for i in range(n_assets):
    for j in range(i+1, n_assets):
        coef = quadratic_zz[i, j] + quadratic_zz[j, i]
        if abs(coef) > 1e-8:
            pauli_str = ['I'] * n_assets
            pauli_str[n_assets - 1 - i] = 'Z'
            pauli_str[n_assets - 1 - j] = 'Z'
            obs_terms.append(("".join(pauli_str), coef))

cost_hamiltonian = SparsePauliOp.from_list(obs_terms)
print("Cost Hamiltonian:\n", cost_hamiltonian)


## 6. Introduction to QAOA

QAOA builds a trial solution by alternating between two ideas:

- The **cost Hamiltonian** $H_C$ gives lower energy to better portfolios.
- The **mixer Hamiltonian** $H_M$ helps the circuit explore different bitstrings instead of getting stuck too early.

One QAOA layer applies the cost operation and then the mixer operation. The number of layers is called $p$. A larger $p$ can represent more complex search behavior, but it also adds more parameters for the classical optimizer to tune.


## 7. Mixer Hamiltonian

The mixer Hamiltonian used here is:
$$ H_M = \sum_i X_i $$

The Pauli-X operation flips qubits between 0 and 1. In portfolio language, it helps the circuit move between different asset-selection choices.

This is why QAOA starts with Hadamard gates first: the circuit begins in a superposition of all possible portfolios, then the cost and mixer operations gradually reshape that superposition toward lower-cost choices.


In [ ]:
# Mixer Hamiltonian consists of Pauli X on each qubit
mixer_terms = []
for i in range(n_assets):
    pauli_str = ['I'] * n_assets
    pauli_str[n_assets - 1 - i] = 'X'
    mixer_terms.append(("".join(pauli_str), 1.0))

mixer_hamiltonian = SparsePauliOp.from_list(mixer_terms)
print("Mixer Hamiltonian:\n", mixer_hamiltonian)


## 8. Parameterized QAOA Circuit

This section builds the actual QAOA ansatz circuit.

The circuit has two sets of trainable angles:
- `gamma` controls how strongly the circuit applies the cost Hamiltonian;
- `beta` controls how strongly the circuit applies the mixer Hamiltonian.

For each QAOA layer, the code applies cost-related rotations using `RZ` and `CX` gates, then applies mixer rotations using `RX` gates. The classical optimizer will later search for the `gamma` and `beta` values that produce the lowest expected cost.


In [ ]:
def create_qaoa_circuit(p):
    circ = QuantumCircuit(n_assets)
    for i in range(n_assets):
        circ.h(i)
        
    gamma_params = [Parameter(f"gamma_{k}") for k in range(p)]
    beta_params = [Parameter(f"beta_{k}") for k in range(p)]
    
    for k in range(p):
        for pauli, coef in cost_hamiltonian.to_list():
            val = coef.real
            z_idx = [i for i, c in enumerate(reversed(pauli)) if c == 'Z']
            if len(z_idx) == 1:
                circ.rz(2 * gamma_params[k] * val, z_idx[0])
            elif len(z_idx) == 2:
                circ.cx(z_idx[0], z_idx[1])
                circ.rz(2 * gamma_params[k] * val, z_idx[1])
                circ.cx(z_idx[0], z_idx[1])
        
        for i in range(n_assets):
            circ.rx(2 * beta_params[k], i)
            
    return circ, gamma_params, beta_params

p_layers = 2
qaoa_circ, gammas, betas = create_qaoa_circuit(p_layers)


## 9. Expectation Value Calculation

The optimizer needs a single number that says how good the current circuit parameters are. That number is the expectation value:
$$ \langle \psi | H_C | \psi \rangle $$

Here, $|\psi\rangle$ is the state created by the QAOA circuit after choosing specific `gamma` and `beta` values. The expectation value is the average cost we would see if we measured that circuit many times.

Because the original problem was written as a minimization problem, the classical optimizer tries to make this expectation value as small as possible.


In [ ]:
def expectation_value(params):
    p = len(params) // 2
    gamma_vals = params[:p]
    beta_vals = params[p:]
    
    bind_dict = {gammas[i]: gamma_vals[i] for i in range(p)}
    bind_dict.update({betas[i]: beta_vals[i] for i in range(p)})
        
    bound_circ = qaoa_circ.assign_parameters(bind_dict)
    sv = Statevector.from_instruction(bound_circ)
    return sv.expectation_value(cost_hamiltonian).real


## 10. Classical Optimizer Setup

QAOA is hybrid: the quantum circuit prepares candidate solutions, and a classical optimizer updates the circuit angles.

This cell creates a random starting point for all QAOA parameters. Since `p_layers = 2`, there are two `gamma` values and two `beta` values, so the optimizer starts with four numbers.

The notebook uses COBYLA because it is a derivative-free optimizer. That means it can improve the parameters without needing gradients from the quantum circuit.


In [ ]:
initial_params = np.random.rand(2 * p_layers) * np.pi
print("Initial Expectation Value:", expectation_value(initial_params))


## 11. Run QAOA (Optimization Loop)

This is the main training loop.

At each step, COBYLA proposes a new set of QAOA parameters. The notebook builds the corresponding circuit, simulates the statevector, computes the expectation value, and sends that cost back to COBYLA.

After the maximum number of iterations, `res.x` stores the best parameters found during the search.


In [ ]:
res = opt.minimize(expectation_value, initial_params, method='COBYLA', options={'maxiter': 200})
print("Optimization Result:\n", res)
optimal_params = res.x


## 12. Retrieve Optimal Portfolio

Once the optimizer finds good parameters, we use them to run the final QAOA circuit and inspect the probability of each bitstring.

Each bitstring is a possible portfolio. Qiskit prints bitstrings with qubit 0 on the right, and this notebook maps qubit 0 to asset 1. For example, with three assets:
- `000` means no assets are selected;
- `101` means asset 1 and asset 3 are selected, while asset 2 is not selected.

The highest-probability bitstrings are the portfolios that the optimized QAOA circuit favors. They are candidate answers, not a mathematical guarantee of the global optimum.


In [ ]:
def get_probabilities(params):
    p = len(params) // 2
    bind_dict = {gammas[i]: params[i] for i in range(p)}
    bind_dict.update({betas[i]: params[i+p] for i in range(p)})
        
    bound_circ = qaoa_circ.assign_parameters(bind_dict)
    sv = Statevector.from_instruction(bound_circ)
    return sv.probabilities_dict()

probs = get_probabilities(optimal_params)
sorted_probs = dict(sorted(probs.items(), key=lambda item: item[1], reverse=True))

print("Top Portfolios by Probability:")
for k, v in list(sorted_probs.items())[:5]:
    print(f"Portfolio {k} : {v*100:.2f}%")


## 13. Visualization & Conclusion

The final plot makes the QAOA result easier to read. Each bar is one possible portfolio bitstring, and the bar height shows how likely the optimized circuit is to output that portfolio.

A good result usually has more probability concentrated on a small number of low-cost portfolios. If the probabilities are spread almost evenly, the circuit or optimizer may need more layers, more iterations, better initial parameters, or a stronger problem formulation.

For learning purposes, try changing `mu`, `Sigma`, `q`, `p_layers`, and `maxiter` to see how the recommended portfolios change.


In [ ]:
states = list(sorted_probs.keys())
probabilities = list(sorted_probs.values())

plt.figure(figsize=(10, 5))
plt.bar(states, probabilities, color='teal')
plt.xlabel("Portfolio (Bitstring)")
plt.ylabel("Probability")
plt.title("QAOA - Optimal Portfolio Probabilities")
plt.xticks(rotation=45)
plt.show()
